In [2]:
import torch

B,T,C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [2]:
# We need bag of words that is mean of previous tokens with current one and this become feature vector
xbow = torch.zeros(B, T, C)
for b in range(B):
  for t in range(T):
    xprev = x[b, :t+1]
    xbow[b, t] = xprev.mean(dim=0)

In [3]:
# implement matrix trick
# We have x - B, T, C
a = torch.tril(torch.ones(T, T))
a = a / a.sum(dim=1, keepdim=True)
xbow2 = a @ x # (T, T) [broadcasting to (B, T, T)] @ (B, T, C ) --> B, T, C
torch.allclose(xbow, xbow2) # xbow and xbow2 is equavalent but xbow2 aproch is optimal

True

In [4]:
# v3 : using softmax its similar to tensor trick it just use softmax for normalize lower triangular matrix at dimenstion 1
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros_like(tril)
wei = torch.masked_fill(wei, tril == 0, float("-inf")) # tril == 0 -> -inf and tril == 1 -> 0
wei = torch.softmax(wei, dim=1) # e(0) -> 1 & e(-inf) -> 0 and softmax normalize them on dimension = 1
xbow3 = wei @ x
torch.allclose(xbow2, xbow3) # it is identical to xbow and xbow2

True

In [7]:
# v4: self attention
B,T,C = 4, 8, 32
x = torch.randn(B, T, C)

tril = torch.tril(torch.ones(T, T))
# Single Head
head_size = 32
queries = torch.nn.Linear(C, head_size, bias=False)
keys = torch.nn.Linear(C, head_size, bias=False)
values = torch.nn.Linear(C, head_size, bias=False)
q = queries(x) # B, T, head_size
k = keys(x)    # B, T, head_size
v = values(x)  # B, T, head_size
wei = q @ k.transpose(-2, -1)  # B, T, head_size @ B, T, head_size [transpose -> B, head_size, T] ----> B, T, T
# wei = torch.zeros_like(tril) # it should not treat every token equally
wei = torch.masked_fill(wei, tril == 0, float("-inf")) # tril == 0 -> -inf and tril == 1 -> 0
wei = torch.softmax(wei, dim=1) # e(0) -> 1 & e(-inf) -> 0 and softmax normalize them on dimension = 1
xbow3 = wei @ v # B, T, T @ B, T, head_size --> B, T, head_size
xbow3.shape

torch.Size([4, 8, 32])